In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.run.model.run_hmm_mamba_classifier import RunMambaClassifier
from src.run.preprocess.config import _project_path

In [ ]:
# CONFIG

DATA_ROOT = _project_path("PaHaW/PaHaW/PaHaW_public")
METADATA_PATH = _project_path("PaHaW/PaHaW/PaHaW_files/corpus_PaHaW.xlsx")

OUTPUT_ROOT = _project_path(
    "experiments/hmm_mamba_classifier/Task_8/xy_acceleration_velocity_pressure_derivative"
)

HMM_STATES = [2, 3, 4, 5]

results = []


In [7]:
for n_states in HMM_STATES:

    print("=" * 80)
    print(f"Running experiment with HMM states = {n_states}")
    print("=" * 80)

    output_dir = OUTPUT_ROOT / f"states_{n_states}"

    runner = RunMambaClassifier(

        data_root=DATA_ROOT,
        metadata_path=METADATA_PATH,

        tasks=[8],

        feature_names=("x", "y"),
        engineered_features=("acceleration", "velocity", "pressure_derivative"),

        n_states=n_states,
        covariance_type="diag",
        n_iter=100,
        tol=1e-2,

        pad_to=128,
        pad_value=0.0,

        batch_size=8,

        epochs=50,

        learning_rate=1e-3,
        regularization=1e-4,

        n_splits=5,

        normalize=True,

        device="cuda" if torch.cuda.is_available() else "cpu",

        output_dir=output_dir,

        training_cfg={

            "metrics": (

                "accuracy",
                "f1",
                "auc",
                "precision",
                "recall",
                "specificity",

            ),

            "results_writer": "csv",

            "save_metrics": True,

            "monitor_metric": "val_auc",

            "early_stopping_patience": 15,

            "min_delta": 1e-4,

            "normalize": True,

            "normalization_eps": 1e-6,

            "label_column": "Disease",

            "positive_value": "PD",

        },

    )

    summary = runner.run()

    row = {
        "states": n_states,
        **summary["averages"],
    }

    results.append(row)

Running experiment with HMM states = 2

START PATIENT-WISE K-FOLD CROSS VALIDATION
Total patients: 75
Folds: 5


FOLD 01/5
Train patients: 60
Validation patients: 15
[WARNING] Mismatch in 00075__8_1.svc: expected 1663, found 1646.
[WARNING] Mismatch in 00075__8_1.svc: expected 1663, found 1646.
[Fold 01] Training Mamba model...
[Fold 01] Epoch 001 | Train loss: 0.5768 | Train acc: 0.7333 | Val loss: 0.4872 | Val acc: 0.8000 | val_auc: 0.8571
[Fold 01] Epoch 002 | Train loss: 0.5435 | Train acc: 0.7333 | Val loss: 0.4794 | Val acc: 0.8000 | val_auc: 0.8571
[Fold 01] Epoch 003 | Train loss: 0.5353 | Train acc: 0.7333 | Val loss: 0.4720 | Val acc: 0.8000 | val_auc: 0.8393
[Fold 01] Epoch 004 | Train loss: 0.5281 | Train acc: 0.7333 | Val loss: 0.4693 | Val acc: 0.8000 | val_auc: 0.8393
[Fold 01] Epoch 005 | Train loss: 0.5198 | Train acc: 0.7333 | Val loss: 0.4797 | Val acc: 0.8000 | val_auc: 0.8393
[Fold 01] Epoch 006 | Train loss: 0.5171 | Train acc: 0.7500 | Val loss: 0.4822 | Val acc:

KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df[
    [
        "states",
        "val_accuracy",
        "val_f1",
        "val_auc",
        "val_precision",
        "val_recall",
        "val_specificity",
        "val_loss",
    ]
]
results_df = results_df.round(4)
results_df

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_ROOT / "comparison_hmm_states.csv"

results_df.to_csv(
    csv_path,
    index=False,
)

print(csv_path)

In [ ]:
best = results_df.loc[results_df["val_auc"].idxmax()]

print("\nBest configuration:")
print(best)

In [ ]:
# ACCURACY
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_accuracy"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation Accuracy")

plt.grid(True)

plt.show()

In [ ]:
# F1
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_f1"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation F1")

plt.grid(True)

plt.show()

In [ ]:
# AUC
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_auc"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation AUC")

plt.grid(True)

plt.show()

In [ ]:
# PRECISION
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_precision"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation Precision")

plt.grid(True)

plt.show()

In [ ]:
# RECALL
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_recall"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation Recall")

plt.grid(True)

plt.show()

In [ ]:
# SPECIFICITY
plt.figure(figsize=(8,5))

plt.plot(
    results_df["states"],
    results_df["val_specificity"],
    marker="o",
)

plt.xlabel("HMM States")
plt.ylabel("Validation Specificity")

plt.grid(True)

plt.show()